In [1]:
import pandas as pd
import os

# Récupère les différents datasets par type de véhicule

In [34]:
path_folder="Data/Type/"
files=os.listdir(path_folder)
files

['4x4.csv',
 'berline.csv',
 'break.csv',
 'cabriolet.csv',
 'coupe.csv',
 'coupecabriolet.csv',
 'monospace.csv',
 'suv.csv',
 'utilitaire.csv']

# Les concatène

In [37]:
df=pd.read_csv(path_folder+files[0]).drop(columns="Unnamed: 0")
df.loc[:,"type"]=files[0][:-4]
for i in files[1:]:
    df_=pd.read_csv(path_folder+i).drop(columns="Unnamed: 0")
    df_.loc[:,"type"]=i[:-4]
    df=pd.concat([df,df_])

In [39]:
df.tail()

,Modele,Marque,Prix,Energie,Kilometre,Annee,Boites de vitesse,Chevaux,options,type
713,Kangoo express,RENAULT,13999,Diesel,51765,2020,Manuelle,95.0,"['cam_recul', 'clim_auto', 'regulateur_vitesse']",utilitaire
714,Jumper,CITROEN,18799,Diesel,80994,2019,Manuelle,NaN,"['cam_recul', 'clim_auto', 'regulateur_vitesse']",utilitaire
715,Berlingo van,CITROEN,11990,Diesel,153772,2021,Manuelle,NaN,"['cam_recul', 'clim_auto', 'regulateur_vitesse']",utilitaire
716,Trafic,RENAULT,24299,Diesel,61175,2022,Manuelle,130.0,"['cam_recul', 'clim_auto', 'regulateur_vitesse']",utilitaire
717,Master,RENAULT,23999,Diesel,55551,2021,Manuelle,135.0,"['cam_recul', 'clim_auto', 'regulateur_vitesse']",utilitaire


In [41]:
df.to_csv("cars_with_type.csv")

In [43]:
df.shape

(15697, 10)

# Prépare le dataset avec celui des véhicule neuf

In [45]:
df2=pd.read_csv("Data/autosphere_neuf.csv").drop(columns='Unnamed: 0').drop_duplicates().rename(columns={"Prix":"PrixNeuf"})

# Moyenne les prix par modèle/Energie/Boite de vitesse

In [48]:
df3=df2.groupby(['Modele', 'Energie',"Boites de vitesse"]).mean()#.shape

In [50]:
df3.head()

PrixNeuf
Modele Energie    Boites de vitesse              
2008   Electrique Automatique        41771.470588
       Essence    Automatique        34482.571429
                  Manuelle           29073.684211
208    Electrique Automatique        37758.472222
       Essence    Automatique        25264.000000

Fusionne les dataframe avec les mêmes clés que celles utilisés précédemment

In [76]:
result = pd.merge(df, df3, how='left', left_on=['Modele', 'Energie',"Boites de vitesse"], right_on=['Modele', 'Energie',"Boites de vitesse"])

In [78]:
result.columns

Index(['Modele', 'Marque', 'Prix', 'Energie', 'Kilometre', 'Annee',
       'Boites de vitesse', 'Chevaux', 'options', 'type', 'PrixNeuf'],
      dtype='object')

In [80]:
result.shape,result.loc[result.PrixNeuf.isna()==False].shape

((15697, 11), (5726, 11))

In [82]:
L=[]
for type in result.type.unique():
    A=result.loc[result.type==type]
    A.loc[:,"Chevaux"]=A.Chevaux.fillna(A.Chevaux.median())
    L+=[A]
df_price_filled=pd.concat(L)

df_price_filled.to_csv("Data/occasion_avec_prix_neuf_a_predire.csv")
df_price_filled.shape

(15697, 11)

# Impute les valeurs manquante (prix neuf et chevaux) en calculant la moyenne par type de véhicule

In [62]:
L=[]
for type in result.type.unique():
    A=result.loc[result.type==type]
    A.loc[:,"PrixNeuf"]=A.PrixNeuf.fillna(A.PrixNeuf.median())
    A.loc[:,"Chevaux"]=A.Chevaux.fillna(A.Chevaux.median())
    L+=[A]
df_price_filled=pd.concat(L)

In [64]:
df_price_filled.isna().sum()

Modele               0
Marque               0
Prix                 0
Energie              0
Kilometre            0
Annee                0
Boites de vitesse    0
Chevaux              0
options              0
type                 0
PrixNeuf             0
dtype: int64

In [65]:
df_price_filled.to_csv("Data/occasion_avec_prix_neuf_filled.csv")

In [69]:
L=[]
result = pd.merge(df, df3, how='inner', left_on=['Modele', 'Energie',"Boites de vitesse"], right_on=['Modele', 'Energie',"Boites de vitesse"])
for type in result.type.unique():
    A=result.loc[result.type==type]
    A.loc[:,"Chevaux"]=A.Chevaux.fillna(A.Chevaux.median())
    L+=[A]
result=pd.concat(L)
result.to_csv("Data/occasion_avec_prix_neuf_inner_join.csv")